[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/ab-test-practice/blob/main/study_notes/week01_study_designing_ab_tests.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 1: Designing A/B Tests

**Course**: A/B Testing in Practice | **Context**: FamilyNest (Airbnb for families with young kids)

**Your role**: Product Data Scientist focused on growing the number of homes on the platform by improving the host onboarding flow.

---

## Setup

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

---

## 1. Choosing Metrics for A/B Tests

When designing an A/B test, you need to decide what you're measuring. Metrics fall into three categories:

### Target Metric (Primary)

The **one metric** you use to make your launch decision. Choose based on:

| Criterion | What it means | Why it matters |
|-----------|--------------|----------------|
| **Sensitivity** | Higher baseline rate = easier to detect changes | A metric with 50% baseline rate has more power than one with 0.1% rate |
| **Timeliness** | How quickly after treatment you can measure it | If the metric takes 90 days to materialize, your test runs 90+ days |
| **Business connection** | Clear link to revenue/growth/retention | Must tie to what the company cares about |

**FamilyNest Example:**

We want to grow homes on the platform. Candidate target metrics:

- **NBL (New Booked Listings)**: A new listing that gets its first booking. Strong business signal, but takes time (host lists -> guest discovers -> guest books).
- **Onboarding completion rate**: How many hosts who start the signup flow complete their listing. More sensitive (higher baseline), more timely (measured in days not weeks).
- **Listing publication rate**: Host publishes their listing. Sits between the two above.

For our onboarding experiment, **listing publication rate** might be the best target metric: it's timely (days), reasonably sensitive, and clearly tied to supply growth.

### Guardrail Metrics

Metrics you monitor to ensure you're not causing unintended harm. You don't optimize for these; you check that they don't degrade.

**FamilyNest guardrail examples:**
- If optimizing listing publication rate, guardrails might be:
  - **Cancellation rate**: Are hosts who onboard faster also canceling more? (rushed through, not ready)
  - **Review score**: Is listing quality degrading?
  - **Time to first response**: Are hosts engaged after onboarding?

### Informative Metrics

Additional metrics you track for learning but don't use for the launch decision.

- Step-by-step completion (which onboarding steps have dropoff?)
- Time spent on each page
- Photo upload count

These help you understand *why* the treatment works (or doesn't) even if they don't determine the ship/no-ship decision.

---

## 2. Hypothesis Formulation

A good hypothesis follows this format:

> **If** [change], **then** [metric] **will** [direction] **because** [mechanism]

**FamilyNest Example:**

> If we simplify the onboarding flow from 8 steps to 4 steps, then the listing publication rate will increase by at least 5%, because reducing friction and cognitive load will help hosts complete the process before losing motivation.

### One-sided vs Two-sided Tests

| Type | Use when | Hypothesis | Alpha split |
|------|----------|------------|-------------|
| **Two-sided** | You want to detect any difference (positive or negative) | H0: p_control = p_treatment | alpha/2 on each tail |
| **One-sided** | You only care about one direction | H0: p_treatment <= p_control | All alpha on one tail |

**In practice**: Most A/B tests use **two-sided tests** because:
1. You want to know if you're making things *worse* too
2. It's more conservative (harder to get significance)
3. Reviewers/stakeholders trust it more

One-sided tests give you more power (easier to detect an effect) but only in one direction.

---

## 3. Sample Size Calculation (Power Analysis)

Before running a test, you need to know: **how many users do I need?**

### Key Parameters

| Parameter | Symbol | Typical value | Meaning |
|-----------|--------|---------------|----------|
| Significance level | alpha | 0.05 | P(reject H0 \| H0 true) - false positive rate |
| Power | 1 - beta | 0.80 | P(reject H0 \| H1 true) - ability to detect real effects |
| Baseline rate | p1 | varies | Current conversion rate |
| Minimum Detectable Effect | MDE | varies | Smallest effect worth detecting |

### Formula for Binary Metrics

Sample size per variant:

$$n = \frac{(Z_{\alpha/2} + Z_{\beta})^2 \cdot [p_1(1-p_1) + p_2(1-p_2)]}{(p_2 - p_1)^2}$$

Where:
- $Z_{\alpha/2}$ = 1.96 for alpha = 0.05 (two-sided)
- $Z_{\beta}$ = 0.84 for power = 0.80
- $p_1$ = baseline conversion rate
- $p_2$ = $p_1 + $ absolute MDE (or $p_1 \times (1 + \text{relative MDE})$)

### Practical Simplification

For small effects where $p_1 \approx p_2$, the formula simplifies to:

$$n \approx \frac{2 \cdot (Z_{\alpha/2} + Z_{\beta})^2 \cdot p(1-p)}{\delta^2}$$

where $p = (p_1 + p_2)/2$ and $\delta = p_2 - p_1$.

In [2]:
def calculate_sample_size(baseline_rate, mde_relative, alpha=0.05, power=0.80, two_sided=True):
    """
    Calculate required sample size per variant for a binary metric.
    
    Parameters:
    -----------
    baseline_rate : float
        Current conversion rate (e.g., 0.10 for 10%)
    mde_relative : float
        Minimum detectable effect as relative change (e.g., 0.05 for 5% relative lift)
    alpha : float
        Significance level (default 0.05)
    power : float
        Statistical power (default 0.80)
    two_sided : bool
        Whether to use two-sided test (default True)
    
    Returns:
    --------
    int : required sample size per variant
    """
    p1 = baseline_rate
    p2 = baseline_rate * (1 + mde_relative)
    
    # Z-scores
    if two_sided:
        z_alpha = norm.ppf(1 - alpha / 2)
    else:
        z_alpha = norm.ppf(1 - alpha)
    z_beta = norm.ppf(power)
    
    # Sample size formula
    numerator = (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2))
    denominator = (p2 - p1) ** 2
    
    n = numerator / denominator
    return int(np.ceil(n))

In [3]:
# FamilyNest Example:
# Current onboarding completion rate: 30%
# We want to detect a 5% relative improvement (30% -> 31.5%)

baseline = 0.30
mde = 0.05  # 5% relative lift

n = calculate_sample_size(baseline_rate=baseline, mde_relative=mde)
print(f"Baseline rate: {baseline:.1%}")
print(f"Expected treatment rate: {baseline * (1 + mde):.1%}")
print(f"Absolute difference: {baseline * mde:.2%}")
print(f"\nRequired sample size per variant: {n:,}")
print(f"Total sample needed (2 variants): {2*n:,}")

Baseline rate: 30.0%
Expected treatment rate: 31.5%
Absolute difference: 1.50%

Required sample size per variant: 14,853
Total sample needed (2 variants): 29,706


In [4]:
# Sensitivity analysis: how does sample size change with MDE?
print("MDE (relative) | Absolute diff | Sample size per variant")
print("-" * 60)
for mde_val in [0.01, 0.02, 0.05, 0.10, 0.15, 0.20]:
    n_val = calculate_sample_size(baseline_rate=0.30, mde_relative=mde_val)
    abs_diff = 0.30 * mde_val
    print(f"  {mde_val:>6.0%}        |    {abs_diff:.3f}     |    {n_val:>10,}")

MDE (relative) | Absolute diff | Sample size per variant
------------------------------------------------------------
      1%        |    0.003     |       367,320
      2%        |    0.006     |        92,086
      5%        |    0.015     |        14,853
     10%        |    0.030     |         3,760
     15%        |    0.045     |         1,690
     20%        |    0.060     |           961


**Key insight**: Smaller effects require exponentially more samples to detect. If you want to detect a 1% relative lift, you need ~25x more users than for a 5% lift. This is why choosing your MDE is a business decision, not just a statistical one - what's the smallest effect that would be worth launching?

---

## 4. Test Duration Calculation

Once you know the sample size, calculate how long the test needs to run:

$$\text{Duration (days)} = \frac{\text{sample\_size\_per\_variant} \times \text{num\_variants}}{\text{daily\_eligible\_traffic}}$$

### Important Considerations

1. **Full weeks**: Always round up to complete weeks to account for weekday/weekend behavioral differences.
2. **Minimum duration**: Even if you have enough traffic in 2 days, run for at least 1-2 weeks to capture time-based variation.
3. **Eligible traffic**: Not all site traffic qualifies for your test. Only count users who would actually see the change.

In [ ]:
def calculate_test_duration(sample_size_per_variant, num_variants, daily_traffic, 
                            traffic_fraction=1.0, round_to_weeks=True):
    """
    Calculate test duration in days.
    
    Parameters:
    -----------
    sample_size_per_variant : int
        Required users per variant
    num_variants : int
        Number of variants (typically 2: control + treatment)
    daily_traffic : int
        Average daily eligible users
    traffic_fraction : float
        Fraction of traffic allocated to the experiment (default 1.0 = 100%)
    round_to_weeks : bool
        Whether to round up to full weeks
    
    Returns:
    --------
    int : test duration in days
    """
    total_sample_needed = sample_size_per_variant * num_variants
    effective_daily = daily_traffic * traffic_fraction
    
    days_needed = total_sample_needed / effective_daily
    
    if round_to_weeks:
        weeks = int(np.ceil(days_needed / 7))
        return weeks * 7
    else:
        return int(np.ceil(days_needed))


# FamilyNest Example:
# 5,000 new users start onboarding per day
# We need ~55,000 per variant (from our earlier calculation with 5% MDE)

sample_per_variant = calculate_sample_size(baseline_rate=0.30, mde_relative=0.05)
daily_users = 5000

duration = calculate_test_duration(
    sample_size_per_variant=sample_per_variant,
    num_variants=2,
    daily_traffic=daily_users
)

print(f"Sample size per variant: {sample_per_variant:,}")
print(f"Total sample needed: {2 * sample_per_variant:,}")
print(f"Daily eligible traffic: {daily_users:,}")
print(f"\nRaw days needed: {2 * sample_per_variant / daily_users:.1f}")
print(f"Rounded to full weeks: {duration} days ({duration // 7} weeks)")

In [ ]:
# What if we only allocate 50% of traffic to the experiment?
duration_half = calculate_test_duration(
    sample_size_per_variant=sample_per_variant,
    num_variants=2,
    daily_traffic=daily_users,
    traffic_fraction=0.50
)
print(f"With 50% traffic allocation: {duration_half} days ({duration_half // 7} weeks)")
print(f"\nTrade-off: slower test but less risk to overall user experience")

---

## 5. Analyzing Results

### Two-Sample T-Test (Z-Test for Proportions)

For binary outcomes (converted/didn't convert), we use a two-proportion z-test.

**Assumptions:**
- Independent observations
- Large enough sample (np > 5 and n(1-p) > 5)
- Random assignment to groups

### P-Value Interpretation

**What p < 0.05 means:**
- If there were truly NO difference between control and treatment, there's less than a 5% probability of observing a difference this large (or larger) by chance alone.

**What p < 0.05 does NOT mean:**
- It does NOT mean there's a 95% chance the treatment works
- It does NOT mean the effect is large or practically meaningful
- It does NOT mean you can't have gotten a false positive

### Confidence Intervals

A 95% CI gives a range of plausible values for the true effect. Much more informative than a p-value alone because it tells you both the direction and magnitude of the effect.

In [ ]:
def get_ci(data, confidence=0.95):
    """
    Calculate the confidence interval for a sample mean.
    
    Parameters:
    -----------
    data : array-like
        Sample data (binary 0/1 for proportions, or continuous)
    confidence : float
        Confidence level (default 0.95)
    
    Returns:
    --------
    tuple : (mean, ci_lower, ci_upper)
    """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    
    z = norm.ppf(1 - (1 - confidence) / 2)
    ci_lower = mean - z * se
    ci_upper = mean + z * se
    
    return mean, ci_lower, ci_upper

In [ ]:
def calculate_results(control_data, treatment_data, confidence=0.95):
    """
    Calculate A/B test results including lift, confidence intervals, and p-value.
    
    Parameters:
    -----------
    control_data : array-like
        Outcomes for control group (0/1 for binary, continuous otherwise)
    treatment_data : array-like
        Outcomes for treatment group
    confidence : float
        Confidence level (default 0.95)
    
    Returns:
    --------
    dict : Results including means, lift, CI, and p-value
    """
    # Basic stats
    n_control = len(control_data)
    n_treatment = len(treatment_data)
    mean_control = np.mean(control_data)
    mean_treatment = np.mean(treatment_data)
    
    # Absolute lift
    abs_lift = mean_treatment - mean_control
    
    # Relative lift
    rel_lift = abs_lift / mean_control if mean_control != 0 else np.nan
    
    # Standard error of the difference
    se_control = np.std(control_data, ddof=1) / np.sqrt(n_control)
    se_treatment = np.std(treatment_data, ddof=1) / np.sqrt(n_treatment)
    se_diff = np.sqrt(se_control**2 + se_treatment**2)
    
    # Confidence interval for the difference
    z = norm.ppf(1 - (1 - confidence) / 2)
    ci_lower = abs_lift - z * se_diff
    ci_upper = abs_lift + z * se_diff
    
    # Two-sided z-test
    z_stat = abs_lift / se_diff
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))
    
    # Statistical significance
    is_significant = p_value < (1 - confidence)
    
    results = {
        'n_control': n_control,
        'n_treatment': n_treatment,
        'mean_control': mean_control,
        'mean_treatment': mean_treatment,
        'absolute_lift': abs_lift,
        'relative_lift': rel_lift,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'se_diff': se_diff,
        'z_statistic': z_stat,
        'p_value': p_value,
        'is_significant': is_significant
    }
    
    return results


def print_results(results):
    """Pretty-print A/B test results."""
    print("=" * 50)
    print("A/B TEST RESULTS")
    print("=" * 50)
    print(f"\nControl:   n={results['n_control']:,}, mean={results['mean_control']:.4f}")
    print(f"Treatment: n={results['n_treatment']:,}, mean={results['mean_treatment']:.4f}")
    print(f"\nAbsolute lift: {results['absolute_lift']:.4f}")
    print(f"Relative lift: {results['relative_lift']:.2%}")
    print(f"\n95% CI for difference: [{results['ci_lower']:.4f}, {results['ci_upper']:.4f}]")
    print(f"P-value: {results['p_value']:.4f}")
    print(f"\nStatistically significant: {'YES' if results['is_significant'] else 'NO'}")
    print("=" * 50)

In [ ]:
# FamilyNest Example: Simulated experiment results
# The simplified onboarding flow experiment ran for 3 weeks

np.random.seed(42)

# Simulate data: control has 30% completion, treatment has 32% (a ~6.7% relative lift)
n_per_group = 15000

control_data = np.random.binomial(1, 0.30, size=n_per_group)
treatment_data = np.random.binomial(1, 0.32, size=n_per_group)

results = calculate_results(control_data, treatment_data)
print_results(results)

In [ ]:
# Interpreting the results:
print("INTERPRETATION")
print("-" * 50)
print(f"""
1. The treatment group (simplified onboarding) had a 
   {results['relative_lift']:.1%} relative improvement over control.

2. The absolute difference was {results['absolute_lift']:.3f} 
   (i.e., {results['absolute_lift']*100:.1f} more completions per 100 users).

3. We are 95% confident the true difference lies between 
   {results['ci_lower']:.4f} and {results['ci_upper']:.4f}.

4. P-value = {results['p_value']:.4f}, which is 
   {'< 0.05 (statistically significant)' if results['p_value'] < 0.05 else '>= 0.05 (not significant)'}.

5. Since the CI {'does not contain' if results['ci_lower'] > 0 else 'contains'} zero,
   we {'can' if results['ci_lower'] > 0 else 'cannot'} reject the null hypothesis.
""")

In [ ]:
# Using get_ci to look at each group individually
control_mean, control_ci_lo, control_ci_hi = get_ci(control_data)
treatment_mean, treatment_ci_lo, treatment_ci_hi = get_ci(treatment_data)

print("Individual group confidence intervals:")
print(f"  Control:   {control_mean:.4f} [{control_ci_lo:.4f}, {control_ci_hi:.4f}]")
print(f"  Treatment: {treatment_mean:.4f} [{treatment_ci_lo:.4f}, {treatment_ci_hi:.4f}]")

---

## 6. Launch Decision Framework

After analyzing results, use this decision framework:

```
                         Target Metric
                    Sig+     Not Sig     Sig-
                  +--------+---------+--------+
Guardrails OK     | LAUNCH | No launch| No launch|
                  +--------+---------+--------+
Guardrails Sig-   |No launch| No launch| No launch|
                  +--------+---------+--------+
```

### Decision Rules:

| Scenario | Decision | Rationale |
|----------|----------|------------|
| Target stat-sig positive AND guardrails not stat-sig negative | **LAUNCH** | Clear win with no harm |
| Target not stat-sig (either direction) | **Don't launch** | Can't prove the change helps; learnings are still valuable |
| Guardrail stat-sig negative | **Don't launch** | Regardless of target metric; you're causing harm |
| Target stat-sig negative | **Don't launch** | The change is making things worse |

### Important Nuances:

- **"Not significant" != "no effect"**: It means we couldn't detect an effect with our sample size. The true effect might exist but be smaller than our MDE.
- **Borderline cases**: If p = 0.06, don't just extend the test hoping to get significance. That's p-hacking. Decide in advance how long to run.
- **Multiple guardrails**: If you test many guardrails, some might be "significant" by chance. Use judgment and look at effect sizes, not just p-values.

In [ ]:
def make_launch_decision(target_results, guardrail_results_list, alpha=0.05):
    """
    Apply the launch decision framework.
    
    Parameters:
    -----------
    target_results : dict
        Output from calculate_results() for the target metric
    guardrail_results_list : list of tuples
        Each tuple is (name, results_dict, direction) where direction is
        'higher_is_better' or 'lower_is_better'
    alpha : float
        Significance threshold
    """
    print("LAUNCH DECISION ANALYSIS")
    print("=" * 50)
    
    # Check target metric
    target_sig = target_results['p_value'] < alpha
    target_positive = target_results['absolute_lift'] > 0
    
    print(f"\nTARGET METRIC:")
    print(f"  Lift: {target_results['relative_lift']:.2%}")
    print(f"  P-value: {target_results['p_value']:.4f}")
    print(f"  Significant & positive: {'YES' if (target_sig and target_positive) else 'NO'}")
    
    # Check guardrails
    guardrail_violated = False
    print(f"\nGUARDRAIL METRICS:")
    for name, g_results, direction in guardrail_results_list:
        g_sig = g_results['p_value'] < alpha
        # Check if the guardrail moved in the bad direction
        if direction == 'higher_is_better':
            bad_direction = g_results['absolute_lift'] < 0
        else:  # lower_is_better (e.g., cancellation rate)
            bad_direction = g_results['absolute_lift'] > 0
        
        violated = g_sig and bad_direction
        if violated:
            guardrail_violated = True
        
        status = "VIOLATED" if violated else "OK"
        print(f"  {name}: lift={g_results['relative_lift']:.2%}, p={g_results['p_value']:.4f} [{status}]")
    
    # Final decision
    print(f"\n{'=' * 50}")
    if guardrail_violated:
        print("DECISION: DO NOT LAUNCH")
        print("Reason: Guardrail metric significantly degraded.")
    elif target_sig and target_positive:
        print("DECISION: LAUNCH")
        print("Reason: Target metric significantly improved, no guardrail violations.")
    else:
        print("DECISION: DO NOT LAUNCH")
        print("Reason: Target metric did not show significant improvement.")


# Example: our onboarding experiment
# Target: listing publication rate (from earlier)
# Guardrail 1: cancellation rate (lower is better)
# Guardrail 2: host response rate (higher is better)

np.random.seed(123)

# Simulate guardrail data (no real change expected)
cancel_control = np.random.binomial(1, 0.05, size=n_per_group)
cancel_treatment = np.random.binomial(1, 0.052, size=n_per_group)  # slight increase, but likely not sig

response_control = np.random.binomial(1, 0.85, size=n_per_group)
response_treatment = np.random.binomial(1, 0.84, size=n_per_group)  # slight decrease

cancel_results = calculate_results(cancel_control, cancel_treatment)
response_results = calculate_results(response_control, response_treatment)

make_launch_decision(
    target_results=results,
    guardrail_results_list=[
        ('Cancellation rate', cancel_results, 'lower_is_better'),
        ('Host response rate', response_results, 'higher_is_better')
    ]
)

---

## 7. SUTVA (Stable Unit Treatment Value Assumption)

### What is SUTVA?

SUTVA states that the treatment assigned to one unit (user) should NOT affect the outcomes of other units. In other words:

> The potential outcome for any user depends only on their own treatment assignment, not on what treatment other users received.

### Why It Matters

If SUTVA is violated, our standard statistical tests become invalid because the observations are no longer independent. The treatment effect we measure may be biased.

### When SUTVA Can Be Violated

| Scenario | Example at FamilyNest | Why it violates SUTVA |
|----------|--------------------------|----------------------|
| **Marketplace/competition effects** | More hosts listing = fewer bookings per host | Treatment group hosts affect control group hosts' booking rates |
| **Network effects** | Host refers another host | Treatment user's behavior directly affects a control user |
| **Shared resources** | Limited featured spots on homepage | If treatment hosts get featured, control hosts get displaced |
| **Social learning** | Hosts in same city share tips | Treatment hosts tell control hosts about the new onboarding flow |

### FamilyNest Specific Example

Suppose our improved onboarding flow successfully increases the number of hosts listing in a specific city (e.g., San Diego). The increased supply could:
- Lower prices for all hosts in that market (hurting control group hosts' revenue)
- Increase overall guest bookings in the market (helping control group too)

In these cases, the control group's outcomes are **contaminated** by the treatment.

### Mitigation Strategies (briefly)

- **Geo-randomization**: Randomize at the city/region level instead of user level
- **Cluster randomization**: Randomize groups of connected users together
- **Design awareness**: Keep the fraction of treated users small enough that spillover is negligible
- **Measure for it**: Compare treatment effects in areas with high vs. low treatment density

---

## Summary: A/B Test Design Checklist

Before running your experiment, confirm:

- [ ] **Hypothesis** written in If/Then/Because format
- [ ] **Target metric** chosen (sensitive, timely, business-connected)
- [ ] **Guardrail metrics** identified
- [ ] **Sample size** calculated via power analysis
- [ ] **Test duration** estimated and rounded to full weeks
- [ ] **SUTVA** considered - will units interfere with each other?
- [ ] **Decision criteria** agreed upon before the test starts
- [ ] **Analysis plan** written (what functions to run, how to interpret)

Remember: the analysis plan should be locked BEFORE you look at results. Changing your approach after seeing data is a recipe for false discoveries.